# ChurnSense-AI — 01 Data Cleaning

**Goal:** Load the IBM Telco Customer Churn dataset, inspect quality issues, clean the data, and export a analysis-ready file to `data/processed/`.

**Paper alignment:** Preprocessing follows telecom churn studies (e.g., Chang et al., 2024 — missing-value checks, train/test split later, churn class imbalance noted).

---
**Before you run:** Place `WA_Fn-UseC_-Telco-Customer-Churn.csv` in `data/raw/` (see `data/raw/README.md`).

## 1. Setup and paths

In [ ]:
# Standard libraries for data cleaning and quick plots
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Plot style (readable defaults for reports)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Project paths — works whether you run from repo root or notebooks/
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file exists: {RAW_PATH.exists()}")

## 2. Load dataset

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {RAW_PATH}.\n"
        "Download from Kaggle (blastchar/telco-customer-churn) and save the CSV in data/raw/."
    )

# Load CSV — customerID is an identifier, not a predictive feature
df = pd.read_csv(RAW_PATH)
print("Dataset loaded successfully.")
df.head()

## 3. Dataset shape and structure

In [ ]:
# Rows = customers, columns = attributes + target (Churn)
n_rows, n_cols = df.shape
print(f"Shape: {n_rows:,} rows × {n_cols} columns")
print(f"\nColumn names:\n{list(df.columns)}")

In [ ]:
# Data types — object columns are usually categorical (Yes/No, contract type, etc.)
df.info()

In [ ]:
# Numeric summary — tenure, charges, SeniorCitizen
df.describe(include="all").T.head(25)

## 4. Missing value analysis

The Telco dataset often has **no standard NaN** in `TotalCharges` — instead, **blank strings** appear for new customers (tenure = 0). We must detect both.

In [ ]:
# Standard pandas missing counts
missing_standard = df.isna().sum()
missing_standard = missing_standard[missing_standard > 0].sort_values(ascending=False)
print("Standard NaN counts (if any):")
print(missing_standard if len(missing_standard) else "None")

In [ ]:
# Hidden missing: empty strings or whitespace-only cells
def count_blank_strings(frame: pd.DataFrame) -> pd.Series:
    """Count blank/whitespace-only values per column (object columns)."""
    blanks = {}
    for col in frame.columns:
        if frame[col].dtype == "object":
            blanks[col] = frame[col].astype(str).str.strip().eq("").sum()
    return pd.Series(blanks).sort_values(ascending=False)

blank_counts = count_blank_strings(df)
blank_counts = blank_counts[blank_counts > 0]
print("Blank string counts:")
print(blank_counts if len(blank_counts) else "None")

In [ ]:
# TotalCharges is stored as text — inspect non-numeric-looking values
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("\nSample TotalCharges values:")
print(df["TotalCharges"].value_counts().head(10))

# Rows where TotalCharges is blank (common when tenure = 0)
blank_total = df["TotalCharges"].astype(str).str.strip().eq("")
print(f"\nRows with blank TotalCharges: {blank_total.sum()}")
if blank_total.any():
    display(df.loc[blank_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head())

## 5. Duplicate handling

In [ ]:
# Duplicate rows (exact match across all columns)
dup_all = df.duplicated().sum()
print(f"Fully duplicate rows: {dup_all}")

# Duplicate customer IDs (should be unique — one row per customer)
dup_id = df["customerID"].duplicated().sum()
print(f"Duplicate customerID values: {dup_id}")

if dup_id > 0:
    display(df[df["customerID"].duplicated(keep=False)].sort_values("customerID").head(10))

## 6. Convert TotalCharges to numeric

**Why:** Models need numbers, not text. New customers with tenure 0 often have empty `TotalCharges`; we impute with `MonthlyCharges` (first-month bill proxy), which is standard in Telco churn projects.

In [ ]:
df_clean = df.copy()

# Strip whitespace from object columns (prevents " Yes" vs "Yes" issues later)
object_cols = df_clean.select_dtypes(include="object").columns
for col in object_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Replace empty strings with NaN before numeric conversion
df_clean["TotalCharges"] = df_clean["TotalCharges"].replace({"": np.nan, "nan": np.nan})
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

missing_after_coerce = df_clean["TotalCharges"].isna().sum()
print(f"TotalCharges NaN after to_numeric: {missing_after_coerce}")

In [ ]:
# Impute missing TotalCharges with MonthlyCharges (reasonable for tenure=0)
mask_missing_total = df_clean["TotalCharges"].isna()
df_clean.loc[mask_missing_total, "TotalCharges"] = df_clean.loc[
    mask_missing_total, "MonthlyCharges"
]

print("Imputation check (tenure=0 rows):")
display(
    df_clean.loc[df_clean["tenure"] == 0, ["tenure", "MonthlyCharges", "TotalCharges"]].head()
)

## 7. Target variable and data quality checks

In [ ]:
# Churn should only be Yes / No
print("Churn value counts:")
print(df_clean["Churn"].value_counts())
print(f"\nChurn rate: {(df_clean['Churn'] == 'Yes').mean():.2%}")

In [ ]:
# SeniorCitizen should be 0 or 1
print("SeniorCitizen unique:", sorted(df_clean["SeniorCitizen"].unique()))

# Tenure and charges should be non-negative
for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    print(f"{col} min: {df_clean[col].min():.2f}, max: {df_clean[col].max():.2f}")

## 8. Remove duplicates (if any)

In [ ]:
rows_before = len(df_clean)

# Drop exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Drop duplicate customer IDs (keep first occurrence)
df_clean = df_clean.drop_duplicates(subset=["customerID"], keep="first")

rows_after = len(df_clean)
print(f"Rows before: {rows_before:,} | after dedup: {rows_after:,} | removed: {rows_before - rows_after}")

## 9. Final missing-value check

In [ ]:
final_missing = df_clean.isna().sum()
final_missing = final_missing[final_missing > 0]
print("Remaining missing values per column:")
print(final_missing if len(final_missing) else "None — dataset is complete.")

## 10. Quick visual — missing values before vs after cleaning

In [ ]:
# Compare TotalCharges validity before/after
total_before = pd.to_numeric(df["TotalCharges"], errors="coerce")
invalid_before = total_before.isna().sum()
invalid_after = df_clean["TotalCharges"].isna().sum()

labels = ["Invalid/blank\n(before)", "Invalid\n(after)"]
values = [invalid_before, invalid_after]

fig, ax = plt.subplots()
ax.bar(labels, values, color=["#e74c3c", "#2ecc71"])
ax.set_ylabel("Count")
ax.set_title("TotalCharges data quality — before vs after cleaning")
for i, v in enumerate(values):
    ax.text(i, v + 0.5, str(v), ha="center")
plt.tight_layout()
plt.show()

## 11. Export cleaned dataset

Downstream notebooks (`02_eda`, feature engineering, modeling) should read from **`data/processed/telco_churn_clean.csv`**.

In [ ]:
OUTPUT_PATH = PROCESSED_DIR / "telco_churn_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {df_clean.shape}")
print(f"\nPreview:")
df_clean.head(3)

## 12. Cleaning summary

| Step | Action |
|------|--------|
| Load | IBM Telco CSV from `data/raw/` |
| Inspect | Shape, dtypes, duplicates, blank strings |
| TotalCharges | Convert to float; impute blanks with `MonthlyCharges` |
| Dedup | Remove duplicate rows / duplicate `customerID` |
| Export | `data/processed/telco_churn_clean.csv` |

**Business note:** Clean billing fields (`TotalCharges`, `MonthlyCharges`) are critical for identifying high-value customers at risk — a core retention KPI for telecom CRM teams.